O pré-processamento típico segue essa ordem:

- Olhar os tipos (df.info()) e ver o que devia ser número/data mas veio como object/str

- Olhar valores únicos (df["COLUNA"].unique()) pra achar sujeira: espaços, formatos inconsistentes.

- Limpar (strip, trocar separador, etc.)

- Converter pro tipo certo

- Só então calcular estatísticas, agrupar, plotar

In [1]:
import pandas as pd

df = pd.read_csv("../dados/brutos/ufc_fight_stats.csv")

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42012 entries, 0 to 42011
Data columns (total 19 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   EVENT       42012 non-null  str    
 1   BOUT        42012 non-null  str    
 2   ROUND       41970 non-null  str    
 3   FIGHTER     41970 non-null  str    
 4   KD          41970 non-null  float64
 5   SIG.STR.    41970 non-null  str    
 6   SIG.STR. %  41970 non-null  str    
 7   TOTAL STR.  41970 non-null  str    
 8   TD          41970 non-null  str    
 9   TD %        41970 non-null  str    
 10  SUB.ATT     41970 non-null  float64
 11  REV.        41970 non-null  float64
 12  CTRL        41970 non-null  str    
 13  HEAD        41970 non-null  str    
 14  BODY        41970 non-null  str    
 15  LEG         41970 non-null  str    
 16  DISTANCE    41970 non-null  str    
 17  CLINCH      41970 non-null  str    
 18  GROUND      41970 non-null  str    
dtypes: float64(3), str(16)
memory usage:

In [3]:
resultados = pd.read_csv("../dados/processados/ufc_fight_results.csv")
chave = ["EVENT", "BOUT"]

# Garante que as chaves estão no mesmo formato nas duas tabelas
df["EVENT"] = df["EVENT"].str.strip()
df["BOUT"] = df["BOUT"].str.strip()

# Confere se a chave é única na tabela de referência (senão o merge duplica linhas)
print("Chave única em resultados?", not resultados.duplicated(chave).any())

antes = len(df)
df = df.merge(resultados[chave], on=chave, how="inner", validate="many_to_one")
print(f"Linhas: {antes} -> {len(df)}")


Chave única em resultados? True
Linhas: 42012 -> 41040


In [4]:
# Conferência: todas as lutas da referência têm estatísticas?
lutas_stats = df[chave].drop_duplicates()
sem_stats = resultados[chave].merge(lutas_stats, on=chave, how="left", indicator=True)
print(sem_stats["_merge"].value_counts())   # o ideal é tudo em "both"


_merge
both          8625
left_only        0
right_only       0
Name: count, dtype: int64


In [33]:
var = 'SIG.STR. %'

df[var].unique()
# df[var].count()

<StringArray>
[ '39%',  '49%',  '47%',  '50%',  '36%',  '52%',  '64%',  '57%',  '44%',
  '40%',  '45%',  '28%',  '41%',   '0%',  '43%',  '71%',  '54%',  '56%',
  '46%',  '62%',  '78%',  '53%',  '69%',  '42%',  '65%',  '11%',  '67%',
  '10%',  '31%',  '30%',  '60%',  '48%',  '66%',  '59%',  '63%',  '23%',
  '58%',  '55%',  '51%',  '25%',  '75%',  '20%',  '34%',  '27%',  '74%',
  '29%',  '37%',  '35%',  '38%',  '33%',  '84%',  '70%',  '80%',  '14%',
  '22%',  '12%',  '26%',  '72%', '100%',  '16%',  '32%',  '92%',  '83%',
  '68%',  '18%',  '---',  '77%',  '61%',  '24%',  '79%',  '85%',  '15%',
  '76%',  '21%',  '19%',  '17%',  '73%',  '82%',  '87%',   '5%',   '8%',
   '7%',  '90%',  '86%',  '81%',  '13%',   '9%',  '89%',  '91%',   '6%',
  '88%',  '95%',  '94%',  '93%',   '4%',   '3%',  '98%',   '2%',  '96%']
Length: 99, dtype: str

In [26]:
df[var]

0        29 of 73
1        40 of 81
2        28 of 59
3         8 of 16
4         8 of 22
           ...   
41035     4 of 12
41036    14 of 33
41037     7 of 13
41038    11 of 22
41039      0 of 4
Name: SIG.STR., Length: 41040, dtype: str

Excluir os nulos e tranformar em Int


In [19]:
df = df.dropna(subset=["FIGHTER"])          # linhas sem lutador = linha vazia
df['KD'] = df['KD'].astype(int)


Uma coluna somente com o número de golpes significativos

In [ ]:
df["SIG.STR."] = df["SIG.STR."].str.split(" of ").str[0].astype(int)

Trocando um texto de '10%' para 0.10

In [51]:
df["SIG.STR. %"] = pd.to_numeric(df["SIG.STR. %"].str.rstrip("%"), errors="coerce") / 100
df["TD %"]       = pd.to_numeric(df["TD %"].str.rstrip("%"), errors="coerce") / 100



AttributeError: Can only use .str accessor with string values, not floating

In [53]:
print(df["TD"].unique())

<StringArray>
[ '0 of 0',  '1 of 3',  '1 of 5',  '3 of 4',  '2 of 7',  '2 of 3',  '1 of 1',
  '1 of 2',  '0 of 2',  '0 of 3',  '0 of 8',  '0 of 1',  '0 of 5',  '0 of 7',
  '3 of 6',  '1 of 6', '1 of 10',  '2 of 2',  '1 of 4',  '2 of 5',  '0 of 6',
 '7 of 12',  '5 of 6', '2 of 10',  '3 of 3',  '0 of 4',  '2 of 4',  '1 of 8',
  '1 of 9',  '2 of 6',  '3 of 5',  '6 of 6',  '4 of 5',  '4 of 7',  '4 of 6',
 '7 of 11', '9 of 12', '5 of 11',  '4 of 4', '3 of 11',  '6 of 7',  '5 of 9',
  '1 of 7',  '3 of 7',  '4 of 9',  '5 of 5',  '7 of 8',  '7 of 9',  '5 of 7',
  '8 of 9',  '2 of 9',  '4 of 8', '2 of 12', '8 of 13',  '3 of 8',  '6 of 8',
  '5 of 8', '0 of 11',  '6 of 9',  '3 of 9', '1 of 11',  '0 of 9',  '2 of 8',
 '1 of 14', '5 of 12', '4 of 11', '5 of 10', '0 of 10', '0 of 12', '8 of 11',
 '8 of 10', '4 of 12', '3 of 10', '6 of 10', '9 of 11', '2 of 11', '1 of 12',
 '3 of 12']
Length: 78, dtype: str
